In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle


In [2]:
## load dataset
df = pd.read_csv('Churn_Modelling.csv.csv')

## preprocess the data 
## drop irrelevant columns
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], axis=1)

## Encode categorical variables
label_encoder_gender = LabelEncoder()
df['Gender'] = label_encoder_gender.fit_transform(df['Gender'])

## oneHotEncode multiple categorical variables
one_hot_encoder_geography = OneHotEncoder()
encoded_geo = one_hot_encoder_geography.fit_transform(df[['Geography']])
encoded_geo_df = pd.DataFrame(encoded_geo.toarray(), columns= one_hot_encoder_geography.get_feature_names_out(['Geography']))
df = pd.concat([df.drop('Geography', axis=1), encoded_geo_df], axis=1)

## separating X and y, train test split, scale the Xs.

X = df.drop('Exited', axis=1)
y= df[['Exited']]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## pickle encoders
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('one_hot_encoder_geography.pkl', 'wb') as file:
    pickle.dump(one_hot_encoder_geography,file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [3]:
# create a function which will create a sequential models which takes neurons and layers as the input
def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation = 'relu', input_shape = (X_train.shape[1],)))

    for i in range(layers-1):
        model.add(Dense(neurons, activation = 'relu'))
    model.add(Dense(1, activation = 'sigmoid'))
    
    model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

    return model


In [4]:
model = KerasClassifier(layers = 1, neurons = 32, build_fn=create_model, epochs=50, batch_size=10, verbose=0)

In [5]:
params_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1,2],
    'epochs': [50, 100]
}

In [6]:
# gridsearch for best neural network
grid = GridSearchCV(estimator=model, param_grid=params_grid, n_jobs=1, cv=4)
grid_result =  grid.fit(X_train, y_train)

d:\GEN-AI COURSE\PROJECTS\ANN CLASSIFICATION\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


d:\GEN-AI COURSE\PROJECTS\ANN CLASSIFICATION\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
d:\GEN-AI COURSE\PROJECTS\ANN CLASSIFICATION\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
d:\GEN-AI COURSE\PROJECTS\ANN CLASSIFICATION\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
d:\GEN-AI COURSE\PROJECTS\ANN CLASSIFICATION\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn

KeyboardInterrupt: 